# Fill-Mask Pipeline (Masked Language Modeling)

## Project Information
- **Source**: Transformers Packt Course (lazyprogrammer.me/course_files/nlp/)
- **Objective**: Predict masked words in text using masked language models
- **Pipeline**: `fill-mask`
- **Model**: BERT, RoBERTa, or similar masked language models

## Overview
This notebook demonstrates how to use the fill-mask pipeline to predict what word should fill a `<mask>` token in a sentence. This is useful for understanding context and word relationships.


In [1]:
%pip install transformers pandas numpy


Note: you may need to restart the kernel to use updated packages.


In [2]:
from transformers import pipeline
import pandas as pd
import numpy as np
import textwrap
import warnings
warnings.filterwarnings('ignore')


c:\Users\helia\Documents\projects\ai-ml\ml-problems\doge-price-prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Initialize Fill-Mask Pipeline


In [ ]:
# Initialize the fill-mask pipeline
mlm = pipeline('fill-mask')
print(f"Pipeline initialized: {type(mlm)}")
print("Using default masked language model (typically BERT or RoBERTa)")


## Basic Fill-Mask Examples


In [ ]:
# Example 1: Simple sentence with mask
text = "The capital of France is <mask>."
results = mlm(text, top_k=5)  # Get top 5 predictions

print(f"Original text: {text}\n")
print("Top 5 predictions:")
for i, result in enumerate(results, 1):
    print(f"{i}. {result['token_str']:15s} (score: {result['score']:.4f}) - {result['sequence']}")


In [ ]:
# Example 2: Business news context
text = "Shares in train and plane-making giant Bombardier have fallen to a 10-year low following the departure of its <mask> and two members of the board."
results = mlm(text, top_k=5)

print(f"Original text: {text}\n")
print("Top 5 predictions:")
for i, result in enumerate(results, 1):
    print(f"{i}. {result['token_str']:15s} (score: {result['score']:.4f})")
    print(f"   Sequence: {result['sequence'][:100]}...")
    print()


## Working with Real Data


In [ ]:
# Load BBC news dataset for context
df = pd.read_csv('bbc_text_cls.csv')
print(f"Dataset shape: {df.shape}")
print(f"Labels: {df['labels'].unique()}")

# Get a sample business article
label = 'business'
texts = df[df['labels'] == label]['text']
np.random.seed(1234)
i = np.random.choice(texts.shape[0])
doc = texts.iloc[i]

print(f"\nSample {label} article (first 300 chars):")
print(textwrap.fill(doc[:300], width=80))


## Automatic Word Replacement Function


In [ ]:
def auto_mask(mask_word, text):
    """
    Automatically replace a word in text with a predicted alternative.
    
    Args:
        mask_word: The word to replace
        text: The original text
    
    Returns:
        Text with the word replaced by the model's prediction
    """
    mlm = pipeline('fill-mask')
    # Replace the word with <mask> token
    text_masked = text.replace(mask_word, "<mask>")
    
    # Get prediction
    result = mlm(text_masked, top_k=1)
    replaced_word = result[0]['token_str'].strip()
    
    # Replace <mask> with predicted word
    text_replaced = text_masked.replace("<mask>", replaced_word)
    
    print(f"Original: {text}")
    print(f"Masked word: '{mask_word}'")
    print(f"Predicted replacement: '{replaced_word}'")
    print(f"Result: {text_replaced}")
    
    return text_replaced

# Example usage
text = "OpenAI is an autoregressive transformer"
result = auto_mask("autoregressive", text)


In [ ]:
# Test different mask positions in the same sentence
base_sentence = "The <mask> scientist discovered a new algorithm."

results = mlm(base_sentence, top_k=3)
print(f"Sentence: {base_sentence}\n")
print("Top 3 predictions:")
for i, result in enumerate(results, 1):
    print(f"{i}. {result['token_str']:15s} (score: {result['score']:.4f})")
    print(f"   {result['sequence']}\n")


## Results Summary

The fill-mask pipeline uses masked language models (like BERT, RoBERTa) to predict what word should fill a masked position in text.

**Key Features:**
- **Contextual Understanding**: Uses bidirectional context to predict words
- **Multiple Predictions**: Returns top-k most likely words with confidence scores
- **Word Relationships**: Can reveal semantic relationships and synonyms
- **Zero-shot**: Works without fine-tuning

**Use Cases:**
- **Text Completion**: Fill in missing words
- **Word Prediction**: Suggest alternatives
- **Language Understanding**: Test model's understanding of context
- **Data Augmentation**: Generate variations of text
- **Spell Checking**: Suggest corrections for masked words

**How it works:**
1. Replace a word with `<mask>` token
2. Model predicts the most likely word for that position
3. Returns top-k predictions with confidence scores
